In [2]:
%pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/48.9 MB 1.5 MB/s eta 0:00:33
    --------------------------------------- 0.8/48.9 MB 1.3 MB/s eta 0:00:38
   - -------------------------------------- 1.3/48.9 MB 1.6 MB/s eta 0:00:31
   - -------------------------------------- 1.6/48.9 MB 1.6 MB/s eta 0:00:30
   - -------------------------------------- 1.8/48.9 MB 1.6 MB/s eta 0:00:30
   - -------------------------------------- 2.1/48.9 MB 1.5 MB/s eta 0:00:32
   -- ------------------------------------- 2.6/48.9 MB 1.5 MB/s eta 0:00:31
   -- ------------------------------------- 2.9/48.9 MB 1.6 MB/s eta 0:00:29
   -- ------------------------------------- 3.4/48.9 MB 1.7 MB/s eta 0:00:28
   --- ------------------------------------ 4.2/48.9 MB 1.8 MB/s eta 0:00:26
   --- -------------


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb

df = pd.read_csv('car_fuel_efficiency_2026.csv')
print(df.shape)
df.head()

(10000, 11)


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [7]:
print(df.isnull().sum())
df = df.fillna(0)
print('\nMissing after fill:', df.isnull().sum().sum())

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

Missing after fill: 0


In [8]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

df_train = df_train.drop(columns='fuel_efficiency_mpg').reset_index(drop=True)
df_val = df_val.drop(columns='fuel_efficiency_mpg').reset_index(drop=True)

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))

print(X_train.shape, X_val.shape)
print(list(dv.get_feature_names_out()))

(6000, 16) (2000, 16)
['acceleration', 'drivetrain=All-wheel drive', 'drivetrain=Front-wheel drive', 'drivetrain=Rear-wheel drive', 'engine_displacement', 'fuel_type=Diesel', 'fuel_type=Gasoline', 'fuel_type=Hybrid', 'horsepower', 'model_year', 'num_cylinders', 'num_doors', 'origin=Asia', 'origin=Europe', 'origin=USA', 'vehicle_weight']


In [9]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

feat = dv.get_feature_names_out()[dt.tree_.feature[0]]
print('Split feature (encoded):', feat)
print('Original column        :', feat.split('=')[0])

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]

Split feature (encoded): model_year
Original column        : model_year


In [10]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
rmse = root_mean_squared_error(y_val, rf.predict(X_val))
print('Validation RMSE:', round(rmse, 4))

Validation RMSE: 1.8332


In [11]:
scores = []
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    rmse = root_mean_squared_error(y_val, rf.predict(X_val))
    scores.append((n, round(rmse, 3)))
    print(n, round(rmse, 3))

best = min(scores, key=lambda t: t[1])
print('\nLowest RMSE (3 d.p.):', best)

10 1.833
50 1.774
100 1.77
150 1.769

Lowest RMSE (3 d.p.): (150, 1.769)


In [12]:
results = []
for d in [10, 15, 20, 25]:
    rmses = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        rmses.append(root_mean_squared_error(y_val, rf.predict(X_val)))
    results.append((d, np.mean(rmses)))
    print(f'max_depth={d:>2}  mean RMSE = {np.mean(rmses):.4f}')

print('\nBest max_depth:', min(results, key=lambda t: t[1])[0])

max_depth=10  mean RMSE = 1.7544
max_depth=15  mean RMSE = 1.7842
max_depth=20  mean RMSE = 1.7868
max_depth=25  mean RMSE = 1.7863

Best max_depth: 10


In [13]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

imp = pd.Series(rf.feature_importances_, index=dv.get_feature_names_out())
print(imp.sort_values(ascending=False).head(10))

candidates = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
print('\nMost important among the 4:', imp[candidates].idxmax())

model_year                      0.296107
vehicle_weight                  0.203313
fuel_type=Gasoline              0.160712
horsepower                      0.078271
engine_displacement             0.061281
acceleration                    0.054998
origin=USA                      0.042152
drivetrain=Front-wheel drive    0.020062
num_doors                       0.019401
fuel_type=Diesel                0.016571
dtype: float64

Most important among the 4: vehicle_weight


In [14]:
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
watchlist = [(dtrain, 'train'), (dval, 'val')]

def train_xgb(eta):
    params = {
        'eta': eta, 'max_depth': 6, 'min_child_weight': 1,
        'objective': 'reg:squarederror', 'nthread': 8,
        'seed': 1, 'verbosity': 1,
    }
    evals = {}
    xgb.train(params, dtrain, num_boost_round=100, evals=watchlist,
              evals_result=evals, verbose_eval=False)
    return evals['val']['rmse']

r03 = train_xgb(0.3)
r01 = train_xgb(0.1)

print('eta=0.3 -> final val RMSE:', round(r03[-1], 4), '| best:', round(min(r03), 4))
print('eta=0.1 -> final val RMSE:', round(r01[-1], 4), '| best:', round(min(r01), 4))

eta=0.3 -> final val RMSE: 1.8265 | best: 1.732
eta=0.1 -> final val RMSE: 1.7248 | best: 1.7122


In [15]:
f03, f01 = round(r03[-1], 3), round(r01[-1], 3)
if f03 < f01: print('Best eta: 0.3')
elif f01 < f03: print('Best eta: 0.1')
else: print('Both give equal value')

Best eta: 0.1
